<center><a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a></center>




## 5b. 遷移學習(Transfer Learning)

到目前為止，我們已經在大型資料集上訓練了準確的模型，並且下載了一個無需訓練即可使用的預訓練模型。但如果我們找不到能夠完全符合需求的預訓練模型，而且也沒有足夠大的資料集來從頭訓練模型，該怎麼辦呢？在這種情況下，我們可以使用一種非常有用的技術，稱為遷移學習([transfer learning](https://blogs.nvidia.com/blog/2019/02/07/what-is-transfer-learning/))。

透過遷移學習，我們採用一個預訓練模型，並在與原始訓練任務有一些重疊的任務上重新訓練它。一個很好的類比是一位在某一種媒介（如油畫）上有技巧的藝術家，想要學習在另一種媒介（如炭筆素描）上創作。我們可以想像，他們在油畫時學到的技巧對於學習如何用炭筆素描會非常有價值。

在深度學習中的一個例子是，假設我們有一個非常擅長識別不同類型汽車的預訓練模型，而我們想要訓練一個識別摩托車類型的模型。汽車模型的許多學習內容可能會非常有用，例如識別頭燈和車輪的能力。

當我們沒有大型且多樣化的資料集時，遷移學習特別有用。在這種情況下，從頭開始訓練的模型可能會快速記憶訓練資料，但無法很好地泛化到新資料。透過遷移學習，你可以增加在小型資料集上訓練出準確且強韌性(robust)模型的機會。


## 5b.1 目標(Objectives)

-   準備一個預訓練模型以進行遷移學習(transfer learning)
-   使用你自己的小型資料集在預訓練(pretrained)模型上執行遷移學習(transfer learning)
-   進一步微調模型以獲得更好的性能

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io

import glob
import json
from PIL import Image

import utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()


## 5b.2 個人化的狗狗門

在我們上一個練習中，我們使用了一個預訓練的[ImageNet](http://www.image-net.org/)模型來讓所有的狗進入，但阻擋其他動物。在這個練習中，我們想要創建一個只讓特定狗進入的狗狗門。在這個案例中，我們將為一隻名叫Bo的狗製作一個自動狗狗門，Bo是2009年至2017年間的美國第一狗。在`data/presidential_doggy_door`資料夾中有更多Bo的圖片。

<img src="data/presidential_doggy_door/train/bo/bo_10.jpg">


<img src="data/presidential\_doggy\_door/train/bo/bo\_10.jpg">

我們的挑戰在於預訓練模型並未被訓練來識別這隻特定的狗，而且我們只有30張Bo的圖片。如果我們嘗試使用這30張圖片從頭開始訓練模型，我們會遇到過度擬合(overfitting)和泛化能力差的問題。然而，如果我們從一個擅長檢測狗的預訓練模型開始，我們可以利用這些學習來使用我們較小的資料集獲得對Bo的泛化理解。我們可以使用遷移學習來解決這個挑戰。

### 5b.2.1 下載預訓練模型

[ImageNet torchvision.models](https://pytorch.org/vision/stable/models.html)通常是電腦視覺遷移學習的好選擇，因為它們已經學會了分類各種不同類型的圖像。在這個過程中，它們學會了檢測許多不同類型的特徵([features](https://developers.google.com/machine-learning/glossary#))，這些特徵在圖像識別中可能很有價值。由於ImageNet模型已經學會了檢測動物，包括狗，它特別適合這個檢測Bo的遷移學習任務。

讓我們從下載預訓練模型開始。

In [ ]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

# load the VGG16 network *pre-trained* on the ImageNet dataset
weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

在我們下載時，會有一個重要的區別。ImageNet模型的最後一層是一個具有1000個單元的密集層([dense layer](https://developers.google.com/machine-learning/glossary#dense-layer))，代表資料集中1000個可能的類別。在我們的案例中，我們希望它進行不同的分類：這是Bo還是不是Bo？我們將添加新的層來專門識別Bo。

In [ ]:
vgg_model.to(device)

### 5b.2.2 凍結(Freezing)基礎模型

在我們將新層添加到預訓練模型([pre-trained model](https://developers.google.com/machine-learning/glossary#pre-trained-model))之前，讓我們先採取一個重要步驟：凍結(Freezing)模型的預訓練層(pre-trained layers)。這意味著當我們訓練時，我們不會更新預訓練模型的基礎層。相反，我們只會更新我們為新分類添加在末端的新層。我們凍結初始層是因為我們想要保留從ImageNet資料集訓練中獲得的學習。如果在這個階段解凍(unfrozen)它們，我們很可能會破壞這些寶貴的資訊。稍後會有一個選項可以解凍並訓練這些層，這個過程稱為微調。

凍結基礎層就像在模型上設置[requires_grad_](https://pytorch.org/docs/stable/generated/torch.Tensor.requires_grad.html)為`False`一樣簡單。

In [ ]:
vgg_model.requires_grad_(False)
print("VGG16 Frozen")

### 5b.2.3 添加新層


現在我們可以將新的可訓練層添加到預訓練模型中。它們將從預訓練層獲取特徵，並將其轉換為新資料集上的預測。我們將向模型添加兩層。在之前的課程中，我們創建了自己的自定義模組([custom module](https://pytorch.org/tutorials/beginner/examples_nn/two_layer_net_module.html))。遷移學習模組的工作方式完全相同。我們可以將其用作序列模型([Sequential Model](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html))中的一層。

然後，我們將添加一個線性層(`Linear` layer)，將VGG16的所有1000個輸出連接到1個神經元。

In [ ]:
N_CLASSES = 1

my_model = nn.Sequential(
    vgg_model,
    nn.Linear(1000, N_CLASSES)
)

my_model.to(device)


如果我們想驗證VGG層是否被凍結，我們可以檢查全部的模型參數([parameters](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html))。

In [ ]:
for idx, param in enumerate(my_model.parameters()):
    print(idx, param.requires_grad)

如果我們想讓VGG層可訓練(trainable)，我們可以取`vgg_model`並將`requires_grad_`設置為`True`。

In [ ]:
vgg_model.requires_grad_(True)
print("VGG16 Unfrozen")
for idx, param in enumerate(my_model.parameters()):
    print(idx, param.requires_grad)

但現在，我們只想訓練我們的新層，所以我們將關閉VGG模型的訓練。

In [ ]:
vgg_model.requires_grad_(False)
print("VGG16 Frozen")

### 5.2.4 編譯(Compiling)模型


與我們之前的練習一樣，我們需要設置損失函式(loss function)和指標(metrics)來編譯模型。我們在這裡必須做出一些不同的選擇。在之前的案例中，我們的分類問題有許多類別。因此，我們選擇了類別交叉熵(categorical crossentropy)來計算我們的損失。在這個案例中，我們只有一個二元分類問題（是Bo或不是Bo），所以我們將使用二元交叉熵( [binary crossentropy](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html))。關於兩者之間的差異的更多細節可以在[這裡](https://gombru.github.io/2018/05/23/cross_entropy_loss/)找到。我們還將使用二元準確度(binary accuracy)而不是傳統的準確度。

通過設置`from_logits=True`，我們告知損失函式([loss function](https://gombru.github.io/2018/05/23/cross_entropy_loss/))輸出值未經正規化(normalized)（例如，未經過softmax處理）。

In [ ]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = Adam(my_model.parameters())
my_model = my_model.to(device)

## 5b.3 資料增強(Data Augmentation)


就像在之前的課程中一樣，我們將創建一個自定義資料集([Dataset](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html))來讀取Bo（和非Bo）的圖片。首先，我們將從VGG的`weights`中獲取預處理轉換器(preprocessing transforms)的列表(list)。

In [ ]:
pre_trans = weights.transforms()


### 5b.3.1 資料集(Dataset)


與之前的課程不同，我們不會從DataFrame中讀取，而是直接讀取圖像文件，並根據文件路徑推斷`label`。

In [ ]:
DATA_LABELS = ["bo", "not_bo"] 
    
class MyDataset(Dataset):
    def __init__(self, data_dir):
        self.imgs = []
        self.labels = []
        
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = glob.glob(data_dir + label + '/*.jpg', recursive=True)
            for path in data_paths:
                img = Image.open(path)
                self.imgs.append(pre_trans(img).to(device))
                self.labels.append(torch.tensor(l_idx).to(device).float())


    def __getitem__(self, idx):
        img = self.imgs[idx]
        label = self.labels[idx]
        return img, label

    def __len__(self):
        return len(self.imgs)

### 5b.3.2 資料載入器(DataLoaders)


現在我們有了自定義的Dataset Class，讓我們創建我們的資料載入器([DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#preparing-your-data-for-training-with-dataloaders))。

In [ ]:
n = 32

train_path = "data/presidential_doggy_door/train/"
train_data = MyDataset(train_path)
train_loader = DataLoader(train_data, batch_size=n, shuffle=True)
train_N = len(train_loader.dataset)

valid_path = "data/presidential_doggy_door/valid/"
valid_data = MyDataset(valid_path)
valid_loader = DataLoader(valid_data, batch_size=n)
valid_N = len(valid_loader.dataset)

### 5b.3.3 資料增強(Data Augmentation)


讓我們應用一些資料增強，這樣模型就有更好的機會識別Bo。這次，我們有彩色圖像，所以我們可以充分利用[ColorJitter](https://pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_illustrations.html#colorjitter)。

In [ ]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

random_trans = transforms.Compose([
    transforms.RandomRotation(25),
    transforms.RandomResizedCrop((IMG_WIDTH, IMG_HEIGHT), scale=(.8, 1), ratio=(1, 1)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.2, hue=.2)
])


## 5b.4 訓練循環(Training Loop)

我們將使用與之前大部分相同的訓練循環，但有一些細微的差異。首先，我們的`get_batch_accuracy`函式會有所不同，因為我們使用二元交叉熵([Binary Cross Entropy](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html) )作為我們的損失函式(loss function)。

當我們的模型`output`大於`0`時，將其通過[sigmoid](https://en.wikipedia.org/wiki/Sigmoid_function)函式運行會更接近`1`。當模型`output`小於`0`時，將其通過sigmoid函式運行會更接近`0`。因此，我們只需要檢查模型輸出是否大於([gt](https://pytorch.org/docs/stable/generated/torch.gt.html))`0`，就可以看出我們的預測傾向於哪個類別。

In [ ]:
def get_batch_accuracy(output, y, N):
    zero_tensor = torch.tensor([0]).to(device)
    pred = torch.gt(output, zero_tensor)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N


我們還有一個部分用於印出最後一組梯度(gradients)，以顯示只有我們新添加的層在學習。

In [ ]:
def train(model, check_grad=False):
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = torch.squeeze(model(random_trans(x)))
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    if check_grad:
        print('Last Gradient:')
        for param in model.parameters():
            print(param.grad)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))



取消下面的註釋以查看模型的梯度樣本。因為VGG16以1000個神經元結束，所以有1000個權重連接到下一層的單個神經元。會印出許多數字！

In [ ]:
#train(my_model, check_grad=True)


`validate`函式大部分保持不變：

In [ ]:
def validate(model):
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = torch.squeeze(model(x))

            loss += loss_function(output, y.float()).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))


真相時刻：模型能學會識別Bo嗎？

In [ ]:
epochs = 10

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train(my_model, check_grad=False)
    validate(my_model)

## 結果討論

訓練和驗證準確度都應該相當高。這是一個相當令人驚嘆的結果！我們能夠在一個小型資料集上進行訓練，但由於從ImageNet模型轉移的知識，它能夠達到高準確度並很好地泛化。這意味著它對Bo和不是Bo的寵物有很好的辨識能力。

如果你看到驗證準確度有一些波動，那也沒關係。我們在下一節有一個改進模型的技巧。

## 微調模型(Fine-Tuning the Model)



現在模型的新層已經訓練完成，我們有選擇應用一個最終技巧來改進模型，稱為微調([fine-tuning](https://developers.google.com/machine-learning/glossary#f))。要做到這一點，我們解凍(unfreeze)整個模型，並使用非常小的學習率([learning rate](https://developers.google.com/machine-learning/glossary#learning-rate))再次訓練它。這將導致預訓練的基礎層進行非常小的步驟並稍微調整，從而小幅度地改進模型。VGG16是一個相對較大的模型，所以小的學習率也會防止過度擬合(overfitting)。

請注意，只有在凍結層的模型完全訓練後才進行這一步驟很重要。我們之前添加到模型中的未訓練的線性層是隨機初始化的。這意味著它需要進行相當多的更新才能正確分類圖像。通過反向傳播([backpropagation](https://developers.google.com/machine-learning/glossary#backpropagation))的過程，最後幾層的大初始更新也會導致預訓練層(pre-trained layers)的潛在大更新。這些更新會破壞那些重要的預訓練特徵。然而，現在這些最終層已經訓練完成並收斂，對整個模型的任何更新都會小得多（特別是使用非常小的學習率），不會破壞早期層的特徵。

讓我們嘗試解凍預訓練層，然後微調模型：

In [ ]:
# Unfreeze the base model
vgg_model.requires_grad_(True)
optimizer = Adam(my_model.parameters(), lr=.000001)

In [ ]:
epochs = 2

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train(my_model, check_grad=False)
    validate(my_model)


在這種情況下，我們只會訓練幾個週期(epochs)。因為VGG16是一個如此大的模型，當它在這個資料集上訓練太長時間時可能會過度擬合。

## 檢視預測結果

現在我們已經有了一個訓練良好的模型，是時候為Bo創建我們的狗門了！我們可以先查看模型產生的預測結果。我們將以與上一個狗門相同的方式對圖像進行預處理。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_image(image_path):
    image = mpimg.imread(image_path)
    plt.imshow(image)

In [ ]:
def make_prediction(file_path):
    show_image(file_path)
    image = Image.open(file_path)
    image = pre_trans(image).to(device)
    image = image.unsqueeze(0)
    output = my_model(image)
    prediction = output.item()
    return prediction


在幾張圖像上試試看預測結果：

In [ ]:
make_prediction('data/presidential_doggy_door/valid/bo/bo_20.jpg')

In [ ]:
make_prediction('data/presidential_doggy_door/valid/not_bo/121.jpg')


看起來負數預測值表示是Bo，而正數預測值表示是其他東西。我們可以使用這些資訊來讓我們的狗門只讓Bo進入！

## 實作練習：Bo的狗門

填寫以下程式碼來實現Bo的狗門：

In [ ]:
def presidential_doggy_door(image_path):
    pred = make_prediction(image_path)
    if FIXME:
        print("It's Bo! Let him in!")
    else:
        print("That's not Bo! Stay out!")

## 解答


點擊下方的`...`查看解答。

In [ ]:
# SOLUTION
def presidential_doggy_door(image_path):
    pred = make_prediction(image_path)
    if pred < 0:
        print("It's Bo! Let him in!")
    else:
        print("That's not Bo! Stay out!")


讓我們試試看！

In [ ]:
presidential_doggy_door('data/presidential_doggy_door/valid/not_bo/131.jpg')

In [ ]:
presidential_doggy_door('data/presidential_doggy_door/valid/bo/bo_29.jpg')

## 總結

做得好！透過遷移學習(Transfer Learning)，你已經使用非常小的數據集構建了一個高度準確的模型。這可以是一種極其強大的技術，並且可能是專案成功與失敗之間的關鍵差異。我們希望這些技術能在未來類似情況下幫助到你！

關於遷移學習的豐富資源，可以參考[NVIDIA TAO Toolkit](https://developer.nvidia.com/tlt-getting-started)。

### 清空記憶體

在繼續之前，請執行以下程式碼區塊(Cell)以清空GPU記憶體。

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## 下一步

到目前為止，本工作坊的重點主要是圖像分類(Image Classification)。在下一部分，為了給你提供更全面的深度學習(Deep Learning)入門，我們將轉換方向，處理序列資料(Sequential Data)，這需要一種不同的方法。

<center><a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a></center>


